# Download Openthoughts Math 30K OPSD without a GPU

This notebook downloads the two revision-pinned Parquet shards of `siyanzhao/Openthoughts_math_30k_opsd`. It does not load a model and does not require CUDA.

Downloads are sequential and resumable. Completed shards are reused; interrupted shards continue from their `.part` files. The default destination is `Openthoughts_math_30k_opsd/` under the current working directory, matching `client_math.ipynb`.


In [ ]:
# Papermill parameters
REPO_ID = "siyanzhao/Openthoughts_math_30k_opsd"
REVISION = "1f33e9dc2e8a1c639ca74f8024ad4a9f1f5eae62"
DESTINATION = "Openthoughts_math_30k_opsd"
ENDPOINTS = ["https://hf-mirror.com", "https://huggingface.co"]
CONNECT_TIMEOUT_SECONDS = 30
READ_TIMEOUT_SECONDS = 120
MAX_ATTEMPTS_PER_SHARD = 100
PROGRESS_INTERVAL_SECONDS = 5
CHUNK_SIZE_MIB = 1


## Prepare the CPU-only downloader


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import requests

destination = Path(DESTINATION).expanduser().resolve()
destination.mkdir(parents=True, exist_ok=True)
remote_shard_paths = [
    "data/train-00000-of-00002.parquet",
    "data/train-00001-of-00002.parquet",
]

print(f"Repository:  {REPO_ID}")
print(f"Revision:    {REVISION}")
print(f"Destination: {destination}")
print("Mode:        sequential HTTP Range download with automatic resume")
print("Files:       " + ", ".join(remote_shard_paths), flush=True)


## Download or resume the two Parquet shards

If the connection closes, the current `.part` file is retained and the next attempt resumes from its existing byte offset. Re-run this cell or the entire notebook after any interruption.


In [ ]:
def format_duration(seconds):
    if seconds is None or seconds == float("inf"):
        return "--:--:--"
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def parquet_has_magic(path):
    if not path.is_file() or path.stat().st_size < 8:
        return False
    with path.open("rb") as handle:
        if handle.read(4) != b"PAR1":
            return False
        handle.seek(-4, os.SEEK_END)
        return handle.read(4) == b"PAR1"

def total_size_from_response(response, resume_from):
    content_range = response.headers.get("Content-Range", "")
    match = re.search(r"/(\d+)$", content_range)
    if match:
        return int(match.group(1))
    content_length = response.headers.get("Content-Length")
    if content_length and content_length.isdigit():
        return int(content_length) + (resume_from if response.status_code == 206 else 0)
    return None

def download_shard(remote_path, shard_index, shard_count):
    filename = Path(remote_path).name
    final_path = destination / filename
    part_path = destination / f"{filename}.part"
    if parquet_has_magic(final_path):
        print(f"[REUSE] {shard_index}/{shard_count} {filename}: {final_path.stat().st_size / 2**20:.2f} MiB", flush=True)
        return final_path
    if final_path.exists():
        invalid_path = destination / f"{filename}.invalid"
        final_path.replace(invalid_path)
        print(f"[WARN] Moved invalid completed file to {invalid_path.name}", flush=True)

    session = requests.Session()
    session.headers.update({
        "User-Agent": "openthoughts-math-resumable-downloader/1.0",
        "Accept-Encoding": "identity",
    })
    shard_started = time.time()
    not_found_endpoints = set()
    for attempt in range(1, int(MAX_ATTEMPTS_PER_SHARD) + 1):
        resume_from = part_path.stat().st_size if part_path.exists() else 0
        endpoint = ENDPOINTS[(attempt - 1) % len(ENDPOINTS)].rstrip("/")
        url = f"{endpoint}/datasets/{REPO_ID}/resolve/{REVISION}/{remote_path}?download=true"
        headers = {"Range": f"bytes={resume_from}-"} if resume_from else {}
        print(
            f"[CONNECT] {shard_index}/{shard_count} {filename}: attempt={attempt}/{MAX_ATTEMPTS_PER_SHARD} "
            f"resume_from={resume_from / 2**20:.2f} MiB endpoint={endpoint}",
            flush=True,
        )
        try:
            with session.get(
                url, headers=headers, stream=True, allow_redirects=True,
                timeout=(CONNECT_TIMEOUT_SECONDS, READ_TIMEOUT_SECONDS),
            ) as response:
                if response.status_code == 416 and parquet_has_magic(part_path):
                    os.replace(part_path, final_path)
                    return final_path
                response.raise_for_status()
                if resume_from and response.status_code == 200:
                    print("[RESTART] Server ignored the Range header; restarting this shard.", flush=True)
                    part_path.unlink(missing_ok=True)
                    resume_from = 0
                expected_total = total_size_from_response(response, resume_from)
                mode = "ab" if resume_from else "wb"
                last_report = time.time()
                with part_path.open(mode) as handle:
                    for chunk in response.iter_content(chunk_size=int(CHUNK_SIZE_MIB * 2**20)):
                        if not chunk:
                            continue
                        handle.write(chunk)
                        now = time.time()
                        if now - last_report >= PROGRESS_INTERVAL_SECONDS:
                            downloaded = part_path.stat().st_size
                            elapsed = max(now - shard_started, 1e-6)
                            speed = downloaded / elapsed
                            if expected_total:
                                percent = 100.0 * downloaded / expected_total
                                eta = (expected_total - downloaded) / speed if speed > 0 else None
                                total_text = f"/{expected_total / 2**20:.2f} MiB ({percent:.2f}%)"
                            else:
                                eta = None
                                total_text = " MiB"
                            print(
                                f"[PROGRESS] {filename}: {downloaded / 2**20:.2f}{total_text} "
                                f"speed={speed / 2**20:.2f} MiB/s elapsed={format_duration(elapsed)} "
                                f"eta={format_duration(eta)}",
                                flush=True,
                            )
                            last_report = now
                    handle.flush()
                    os.fsync(handle.fileno())

                downloaded = part_path.stat().st_size
                if expected_total is not None and downloaded < expected_total:
                    raise IOError(f"incomplete response: {downloaded}/{expected_total} bytes")
                if not parquet_has_magic(part_path):
                    raise IOError("downloaded file does not yet contain complete Parquet boundary markers")
                os.replace(part_path, final_path)
                print(f"[DONE] {filename}: {final_path.stat().st_size / 2**20:.2f} MiB", flush=True)
                return final_path
        except (requests.RequestException, OSError) as exc:
            status_code = exc.response.status_code if isinstance(exc, requests.HTTPError) and exc.response is not None else None
            if status_code == 404:
                not_found_endpoints.add(endpoint)
                if len(not_found_endpoints) >= len(ENDPOINTS):
                    raise FileNotFoundError(f"Shard does not exist at the pinned revision: {remote_path}") from exc
                print(f"[ENDPOINT MISS] {endpoint} returned 404; trying the next endpoint.", flush=True)
                continue
            if status_code is not None and 400 <= status_code < 500 and status_code not in (408, 429):
                raise RuntimeError(f"Non-retryable HTTP {status_code} for {remote_path}") from exc
            if attempt >= int(MAX_ATTEMPTS_PER_SHARD):
                raise RuntimeError(f"Failed to download {filename} after {attempt} attempts") from exc
            wait_seconds = min(60, max(2, attempt * 2))
            saved = part_path.stat().st_size / 2**20 if part_path.exists() else 0.0
            print(f"[RETRY] {type(exc).__name__}: {exc}; saved={saved:.2f} MiB; wait={wait_seconds}s", flush=True)
            time.sleep(wait_seconds)
    raise RuntimeError(f"Unreachable retry state for {filename}")

downloaded_paths = [
    download_shard(remote_path, index, len(remote_shard_paths))
    for index, remote_path in enumerate(remote_shard_paths, start=1)
]


## Record the local dataset and show how to use it


In [ ]:
manifest = {
    "repository": REPO_ID,
    "revision": REVISION,
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    "files": [
        {"name": path.name, "bytes": path.stat().st_size}
        for path in downloaded_paths
    ],
}
manifest_path = destination / "dataset_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print("\nDataset ready.")
print(f"Manifest: {manifest_path}")
print(f"Total size: {sum(path.stat().st_size for path in downloaded_paths) / 2**30:.3f} GiB")
print("client_math.ipynb setting:")
print(f'DATASET_LOCAL_DIR = "{destination}"')
